# Notebook 01 — PDF 문서 처리
## TechDocRAG: 멀티모달 기술 문서 RAG
### 이 노트북에서 배울 것
- PyMuPDF로 PDF → 페이지별 고해상도 이미지 변환
- 텍스트 추출 (일반 텍스트 + 폰트/레이아웃 정보)
- pdfplumber로 표(table) 구조 추출 → 마크다운 변환
- 페이지 컨텐츠 타입 자동 분류
- 청킹 전략: 페이지 단위 + 의미 단위 분할

In [ ]:
# 한글 폰트 + 기본 설정
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import os, sys
from pathlib import Path

# 프로젝트 루트
ROOT = Path().absolute().parent
DATA_DIR   = ROOT / 'data' / 'docs'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT      : {ROOT}")
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 1. 샘플 PDF 생성 (데모용)
기술 문서가 없을 경우를 위해 텍스트 + 표 + 이미지가 포함된 샘플 PDF를 자동 생성합니다.

In [ ]:
# 샘플 PDF 생성 — 자동차 기술 문서 (텍스트 + 표 포함)
# reportlab이 없으면 PyMuPDF로 직접 생성
import fitz  # PyMuPDF

SAMPLE_PDF = DATA_DIR / 'sample_ev_manual.pdf'
DATA_DIR.mkdir(parents=True, exist_ok=True)

def create_sample_pdf(path: Path):
    doc = fitz.open()

    # 페이지 1: 개요 텍스트
    page = doc.new_page(width=595, height=842)
    page.insert_text((50, 80),  "EV 배터리 관리 시스템 기술 매뉴얼", fontsize=18, color=(0,0,0))
    page.insert_text((50, 120), "1장. 시스템 개요", fontsize=14)
    page.insert_text((50, 150),
        "본 매뉴얼은 전기차(EV) 배터리 관리 시스템(BMS)의 구조와\n"
        "동작 원리를 설명합니다. BMS는 배터리 셀의 전압, 전류,\n"
        "온도를 실시간으로 모니터링하여 과충전·과방전을 방지합니다.\n\n"
        "핵심 기능:\n"
        "  - 셀 밸런싱 (Cell Balancing)\n"
        "  - 상태 추정 (SOC/SOH Estimation)\n"
        "  - 열 관리 (Thermal Management)\n"
        "  - 고장 진단 (Fault Diagnosis)",
        fontsize=11, color=(0.1, 0.1, 0.1))

    # 페이지 2: 사양 표
    page2 = doc.new_page(width=595, height=842)
    page2.insert_text((50, 60), "2장. 배터리 사양", fontsize=14)
    page2.insert_text((50, 95), "표 1. EV 배터리 팩 주요 사양", fontsize=12)

    # 간단한 표 그리기
    table_data = [
        ["항목",          "사양",          "단위"],
        ["공칭 전압",     "400",           "V"],
        ["배터리 용량",   "77.4",          "kWh"],
        ["최대 충전 전력", "350",           "kW"],
        ["셀 화학",       "NCM 811",       "-"],
        ["동작 온도",     "-30 ~ +55",     "°C"],
        ["사이클 수명",   ">2,000",        "cycle"],
    ]
    x0, y0, row_h, col_w = 50, 120, 28, [160, 140, 80]
    for r, row in enumerate(table_data):
        y = y0 + r * row_h
        for c, cell in enumerate(row):
            x = x0 + sum(col_w[:c])
            rect = fitz.Rect(x, y, x + col_w[c], y + row_h)
            color = (0.85, 0.90, 0.95) if r == 0 else (1, 1, 1)
            page2.draw_rect(rect, color=color, fill=color)
            page2.draw_rect(rect, color=(0.5, 0.5, 0.5), width=0.5)
            page2.insert_text((x + 5, y + row_h - 8), cell, fontsize=10)

    page2.insert_text((50, 340),
        "3장. SOC 추정 알고리즘\n\n"
        "SOC(State of Charge)는 배터리 잔존 용량을 나타내며\n"
        "칼만 필터(Extended Kalman Filter) 기반으로 추정합니다.\n\n"
        "SOC = SOC_0 - (1/Q_max) * ∫i(t)dt\n\n"
        "여기서 Q_max는 최대 용량, i(t)는 순간 전류입니다.",
        fontsize=11)

    # 페이지 3: 진단 코드
    page3 = doc.new_page(width=595, height=842)
    page3.insert_text((50, 60), "4장. 고장 진단 코드 (DTC)", fontsize=14)
    page3.insert_text((50, 95), "표 2. 주요 배터리 DTC 코드", fontsize=12)

    dtc_data = [
        ["DTC 코드",  "설명",                 "심각도"],
        ["P0A0F",    "배터리 팩 전압 이상",   "HIGH"],
        ["P0A1E",    "배터리 온도 과열",       "HIGH"],
        ["P0A80",    "셀 밸런싱 실패",         "MEDIUM"],
        ["P0AFA",    "SOC 추정 오류",          "LOW"],
        ["P1A00",    "냉각 시스템 이상",       "MEDIUM"],
    ]
    x0, y0, row_h, col_w = 50, 120, 28, [100, 200, 100]
    for r, row in enumerate(dtc_data):
        y = y0 + r * row_h
        for c, cell in enumerate(row):
            x = x0 + sum(col_w[:c])
            rect = fitz.Rect(x, y, x + col_w[c], y + row_h)
            if r == 0:
                fill = (0.85, 0.90, 0.95)
            elif 'HIGH' in row:
                fill = (1.0, 0.92, 0.92)
            elif 'MEDIUM' in row:
                fill = (1.0, 0.97, 0.90)
            else:
                fill = (1, 1, 1)
            page3.draw_rect(rect, color=fill, fill=fill)
            page3.draw_rect(rect, color=(0.5, 0.5, 0.5), width=0.5)
            page3.insert_text((x + 5, y + row_h - 8), cell, fontsize=10)

    page3.insert_text((50, 320),
        "5장. 안전 지침\n\n"
        "배터리 팩 작업 시 반드시 아래 절차를 따르십시오:\n"
        "1. 서비스 플러그(Service Plug)를 분리하여 고전압 차단\n"
        "2. 절연 장갑(Class E, 1000V 이상) 착용\n"
        "3. 작업 전 전압 측정기로 잔류 전압 확인\n"
        "4. 고전압 케이블 커넥터 분리 순서: 부(-) → 정(+)\n"
        "5. 작업 완료 후 절연 저항 측정 (>1MΩ)",
        fontsize=11)

    doc.save(str(path))
    print(f"샘플 PDF 생성 완료: {path} ({doc.page_count}페이지)")

if not SAMPLE_PDF.exists():
    create_sample_pdf(SAMPLE_PDF)
else:
    print(f"기존 파일 사용: {SAMPLE_PDF}")

## 2. PDF → 페이지 이미지 변환
각 페이지를 고해상도 PNG로 렌더링합니다. 해상도가 높을수록 Vision LLM이 표·다이어그램을 더 잘 인식합니다.

In [ ]:
# PDF → 페이지 이미지 변환
from PIL import Image
import io

def pdf_to_images(pdf_path: Path, output_dir: Path, dpi: int = 200) -> list[Path]:
    """
    PDF 각 페이지를 PNG 이미지로 변환.
    dpi=200 → A4 기준 1654×2339px (Vision LLM 품질 충분)
    """
    doc = fitz.open(str(pdf_path))
    doc_name = pdf_path.stem
    img_dir  = output_dir / doc_name
    img_dir.mkdir(parents=True, exist_ok=True)

    image_paths = []
    scale = dpi / 72  # PyMuPDF 기본 72dpi 기준
    mat   = fitz.Matrix(scale, scale)

    for page_num, page in enumerate(doc):
        pix  = page.get_pixmap(matrix=mat, alpha=False)
        path = img_dir / f"page_{page_num:03d}.png"
        pix.save(str(path))
        image_paths.append(path)

    doc.close()
    print(f"[{doc_name}] {len(image_paths)}페이지 → {img_dir}")
    return image_paths

# 변환 실행
image_paths = pdf_to_images(SAMPLE_PDF, OUTPUT_DIR, dpi=200)
print(f"\n총 {len(image_paths)}개 이미지 생성됨")

In [ ]:
# 변환된 이미지 미리보기
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, len(image_paths), figsize=(6 * len(image_paths), 8))
if len(image_paths) == 1:
    axes = [axes]

for ax, img_path in zip(axes, image_paths):
    img = mpimg.imread(str(img_path))
    ax.imshow(img)
    ax.set_title(f"Page {img_path.stem.split('_')[-1]}", fontsize=12)
    ax.axis('off')

plt.suptitle('PDF 페이지 이미지 변환 결과', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print(f"이미지 크기: {img.shape} (H×W×C)")

## 3. 텍스트 추출
PyMuPDF의 블록 단위 추출로 텍스트의 위치·폰트 크기 정보를 함께 가져와 제목/본문 구분에 활용합니다.

In [ ]:
# PyMuPDF 텍스트 추출 — 블록 단위 (폰트 크기 포함)
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TextBlock:
    page_num:   int
    text:       str
    font_size:  float
    is_heading: bool  # 폰트 크기가 평균보다 크면 제목으로 분류
    bbox:       tuple  # (x0, y0, x1, y1)

def extract_text_blocks(pdf_path: Path) -> list[TextBlock]:
    doc = fitz.open(str(pdf_path))
    all_blocks = []

    for page_num, page in enumerate(doc):
        # dict 모드로 폰트 크기 포함한 블록 추출
        blocks = page.get_text("dict")["blocks"]

        for block in blocks:
            if block.get("type") != 0:  # 0=텍스트, 1=이미지
                continue

            for line in block.get("lines", []):
                text_parts, font_sizes = [], []
                for span in line.get("spans", []):
                    t = span["text"].strip()
                    if t:
                        text_parts.append(t)
                        font_sizes.append(span["size"])

                text = " ".join(text_parts).strip()
                if not text:
                    continue

                avg_size = sum(font_sizes) / len(font_sizes) if font_sizes else 11
                all_blocks.append(TextBlock(
                    page_num=page_num,
                    text=text,
                    font_size=round(avg_size, 1),
                    is_heading=False,  # 나중에 전체 평균 기준으로 재분류
                    bbox=tuple(block["bbox"])
                ))

    # 전체 평균 폰트 크기 기준으로 제목/본문 분류
    if all_blocks:
        mean_size = sum(b.font_size for b in all_blocks) / len(all_blocks)
        for b in all_blocks:
            b.is_heading = b.font_size > mean_size * 1.2

    doc.close()
    return all_blocks

text_blocks = extract_text_blocks(SAMPLE_PDF)
print(f"총 텍스트 블록: {len(text_blocks)}개")
print(f"제목 블록: {sum(b.is_heading for b in text_blocks)}개")
print(f"\n--- 첫 10개 블록 ---")
for b in text_blocks[:10]:
    label = "[제목]" if b.is_heading else "[본문]"
    print(f"  p{b.page_num} {label} (fs={b.font_size}): {b.text[:60]}")

## 4. 표(Table) 추출 — pdfplumber
pdfplumber는 표의 셀 경계선을 인식하여 구조화된 데이터로 추출합니다. 표를 마크다운으로 변환하면 LLM이 더 잘 이해합니다.

In [ ]:
# pdfplumber로 표 추출 → 마크다운 변환
import pdfplumber

def table_to_markdown(table: list[list]) -> str:
    """2D 리스트 표를 마크다운 테이블 문자열로 변환"""
    if not table or not table[0]:
        return ""

    # None 셀 처리
    rows = [[str(cell).strip() if cell else "" for cell in row] for row in table]

    # 컬럼 너비 계산
    col_widths = [max(len(rows[r][c]) for r in range(len(rows)))
                  for c in range(len(rows[0]))]

    lines = []
    for i, row in enumerate(rows):
        line = "| " + " | ".join(cell.ljust(col_widths[c]) for c, cell in enumerate(row)) + " |"
        lines.append(line)
        if i == 0:  # 헤더 구분선
            sep = "| " + " | ".join("-" * col_widths[c] for c in range(len(row))) + " |"
            lines.append(sep)

    return "\n".join(lines)


def extract_tables(pdf_path: Path) -> list[dict]:
    """PDF에서 표 추출 → 마크다운으로 변환"""
    tables_info = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for t_idx, table in enumerate(tables):
                md = table_to_markdown(table)
                if md:
                    tables_info.append({
                        'page_num': page_num,
                        'table_idx': t_idx,
                        'markdown': md,
                        'rows': len(table),
                        'cols': len(table[0]) if table else 0
                    })
    return tables_info


tables = extract_tables(SAMPLE_PDF)
print(f"추출된 표: {len(tables)}개\n")
for t in tables:
    print(f"[페이지 {t['page_num']}] 표 {t['table_idx']+1} ({t['rows']}행×{t['cols']}열)")
    print(t['markdown'])
    print()

## 5. 페이지 컨텐츠 타입 분류 + 청킹
각 페이지를 텍스트/표/이미지 비중에 따라 분류하고, RAG에 적합한 청크 단위로 분할합니다.

In [ ]:
# 페이지별 컨텐츠 타입 분류 + 청크 생성
from dataclasses import dataclass

@dataclass
class PageChunk:
    doc_name:   str
    page_num:   int
    chunk_id:   str         # "{doc}_{page:03d}"
    text:       str         # 텍스트 + 마크다운 표 통합
    image_path: str         # 대응하는 페이지 PNG 경로
    content_type: str       # 'text_heavy' | 'table_heavy' | 'mixed'
    has_table:  bool
    char_count: int


def build_page_chunks(
    pdf_path:    Path,
    image_paths: list[Path],
    text_blocks: list[TextBlock],
    tables:      list[dict]
) -> list[PageChunk]:
    """페이지 단위로 텍스트 + 표를 통합한 청크를 생성"""
    doc_name = pdf_path.stem
    doc      = fitz.open(str(pdf_path))
    chunks   = []

    # 페이지별 표 인덱스
    page_tables: dict[int, list[dict]] = {}
    for t in tables:
        page_tables.setdefault(t['page_num'], []).append(t)

    # 페이지별 텍스트 블록 인덱스
    page_blocks: dict[int, list[TextBlock]] = {}
    for b in text_blocks:
        page_blocks.setdefault(b.page_num, []).append(b)

    for page_num in range(len(doc)):
        # 텍스트 조립 (제목은 ## 표시)
        blocks    = page_blocks.get(page_num, [])
        text_parts = []
        for b in blocks:
            if b.is_heading:
                text_parts.append(f"## {b.text}")
            else:
                text_parts.append(b.text)
        page_text = "\n".join(text_parts)

        # 표 마크다운 추가
        t_list    = page_tables.get(page_num, [])
        table_mds = [t['markdown'] for t in t_list]
        if table_mds:
            page_text += "\n\n" + "\n\n".join(table_mds)

        # 컨텐츠 타입 분류
        text_len  = len(page_text)
        table_len = sum(len(t['markdown']) for t in t_list)
        if table_len > text_len * 0.5:
            ctype = 'table_heavy'
        elif text_len > 200:
            ctype = 'text_heavy'
        else:
            ctype = 'mixed'

        img_path = str(image_paths[page_num]) if page_num < len(image_paths) else ""

        chunks.append(PageChunk(
            doc_name=doc_name,
            page_num=page_num,
            chunk_id=f"{doc_name}_p{page_num:03d}",
            text=page_text.strip(),
            image_path=img_path,
            content_type=ctype,
            has_table=len(t_list) > 0,
            char_count=len(page_text)
        ))

    doc.close()
    return chunks


chunks = build_page_chunks(SAMPLE_PDF, image_paths, text_blocks, tables)
print(f"생성된 청크: {len(chunks)}개\n")
for c in chunks:
    print(f"  [{c.chunk_id}] type={c.content_type}, has_table={c.has_table}, chars={c.char_count}")
    print(f"    텍스트 미리보기: {c.text[:80].replace(chr(10), ' ')}...")

In [ ]:
# 청크 통계 시각화
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame([{
    'chunk_id':     c.chunk_id,
    'content_type': c.content_type,
    'has_table':    c.has_table,
    'char_count':   c.char_count
} for c in chunks])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 컨텐츠 타입 분포
type_counts = df['content_type'].value_counts()
axes[0].bar(type_counts.index, type_counts.values, color=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('페이지 컨텐츠 타입 분포')
axes[0].set_xlabel('타입')
axes[0].set_ylabel('페이지 수')

# 청크별 텍스트 길이
axes[1].bar(df['chunk_id'], df['char_count'], color='#4C72B0')
axes[1].set_title('청크별 텍스트 길이')
axes[1].set_xlabel('청크 ID')
axes[1].set_ylabel('문자 수')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(f"\n청크 통계:")
print(f"  평균 문자 수: {df['char_count'].mean():.0f}")
print(f"  표 포함 청크: {df['has_table'].sum()}개 / {len(df)}개")

In [ ]:
# NB01 결과 저장 — NB02에서 로드
import json

# PageChunk를 직렬화 가능한 dict로 변환
chunks_data = [{
    'doc_name':     c.doc_name,
    'page_num':     c.page_num,
    'chunk_id':     c.chunk_id,
    'text':         c.text,
    'image_path':   c.image_path,
    'content_type': c.content_type,
    'has_table':    c.has_table,
    'char_count':   c.char_count
} for c in chunks]

save_path = ROOT / 'output' / 'chunks.json'
with open(save_path, 'w', encoding='utf-8') as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

print(f"청크 저장 완료: {save_path}")
print(f"총 {len(chunks_data)}개 청크")
print("\n[다음 단계] Notebook 02 — 멀티모달 임베딩 & ChromaDB 인덱싱")